# Практика 6. Raw-слой в Data Lake: форматы, версии, партиционирование

Стартовый ноутбук:
- выполняйте ячейки сверху вниз, если требуется, то заполняйте TODO
- готовые решения/ответы преподаватель выдаст после занятия
- ноутбук запускается на компьютере, не в Docker.

**Источники** (PostgreSQL `shop`, данные загружаются из `init.sql` при старте compose):
- заказы `orders(order_id, customer_id, order_created_at, status, total_amount)`;
- события `events(event_id, customer_id, event_time, event_type, product_id, session_id)`.

**Конвенция ключей** (один префикс на источник, `load_date` — дата загрузки, не дата из данных):

```text
de-course/
├── raw/events/load_date=<DATE>/part-0000.<ext>
├── raw/orders/load_date=<DATE>/part-0000.<ext>
└── raw_flat/orders/part-*.parquet   # вариант «в лоб»: даты в ключе нет
```

**Воспроизводимые загрузки:** «вчера» = `order_id` 101–120, `event_id` 2411–2413; «сегодня» = всё остальное.

**Справочник по функциям ноутбука**

| Функция | Назначение |
|---|---|
| `encode_frames(df, parquet_compression=None)` | Кодирует DataFrame в три формата (CSV, JSON Lines, Parquet) и возвращает `dict` байтов. `parquet_compression=None` — без сжатия, `'zstd'`/`'snappy'` — с сжатием |
| `upload(source, load_date, df, parquet_compression=None)` | Загружает 3 кодирования `df` в MinIO по ключу `raw/{source}/load_date=<DATE>/part-0000.<fmt>` с метаданными `source/layer/format/rows` |
| `read_csv_bytes` / `read_json_bytes` / `read_parquet_one_col`(enc_orders, col_name) | Читают одну колонку из кодирования: CSV/JSON — весь файл, Parquet — только колонка из row group 0 |
| `bench(fn, enc_orders, col_name, n=5)` | Запускает `fn` `n` раз, возвращает `(минимальное время в мс, результат)` |
| `print_parquet_metadata(enc_orders)` | Выводт на экран метаданные файла Parquet |
| `plan_files(prefix)` | Перечисляет все ключи объектов MinIO по префиксу (с пагинацией) — «план чтения» |

## 1. Окружение и подключение к MinIO

Перед запуском: `docker compose up -d` (postgres 5433, minio 9000/9001). Если VS Code попросит перезапустить kernel после установки пакетов — перезапустите и выполните ячейки сверху вниз.

In [3]:
%pip install -q "psycopg[binary]" boto3 pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [4]:
import boto3
import io
import time
import pandas as pd
import psycopg

from datetime import date, timedelta
from botocore.config import Config

PG = dict(host='localhost', port=5433, dbname='shop', user='student', password='studentpass')
S3_ENDPOINT = 'http://localhost:9000'  # ноутбук -> MinIO с компьютера
S3_USER = 'minioadmin'
S3_PASSWORD = 'minioadmin'
BUCKET = 'de-course'

TODAY = date.today().isoformat()
YESTERDAY = (date.today() - timedelta(days=1)).isoformat()

s3 = boto3.client(
    service_name='s3',
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_USER,
    aws_secret_access_key=S3_PASSWORD,
    region_name='us-east-1',
    config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
)
existing = {item['Name'] for item in s3.list_buckets()['Buckets']}
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)

s3.put_bucket_versioning(
    Bucket=BUCKET,
    VersioningConfiguration={
        'Status': 'Enabled'
    }
)

print(f'Бакет {BUCKET} создан, версионирование включено')
print('MinIO Console http://localhost:9001')

Бакет de-course создан, версионирование включено
MinIO Console http://localhost:9001


## 2. Загрузка: события и заказы из PostgreSQL

Достаём источники **без преобразований** (raw-контракт: «значения не изменены»). Список колонок ниже соответствует `init.sql`.

In [5]:
with psycopg.connect(**PG) as conn:
    events_df = pd.read_sql(
        """
        SELECT event_id, customer_id, event_time, event_type, product_id, session_id
        FROM events ORDER BY event_id
        """, conn)
    orders_df = pd.read_sql(
        """
        SELECT order_id, customer_id, order_created_at, status, total_amount
        FROM orders ORDER BY order_id
        """, conn)
print('events:', events_df.shape, '| orders:', orders_df.shape)
orders_df.head()

events: (387, 6) | orders: (141, 5)


/tmp/ipykernel_155159/2083524462.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  events_df = pd.read_sql(
/tmp/ipykernel_155159/2083524462.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders_df = pd.read_sql(


,order_id,customer_id,order_created_at,status,total_amount
0,101,1,2026-03-01 10:00:00,paid,1000.0
1,102,1,2026-03-01 10:00:00,paid,900.0
2,103,1,2026-03-03 11:00:00,paid,900.0
3,104,2,2026-03-01 11:00:00,paid,1000.0
4,105,2,2026-03-02 12:00:00,paid,900.0


In [6]:
# Разделяем на две загрузки: «вчера» — воспроизводимое подмножество, «сегодня» — остальное
events_y = events_df[events_df['event_id'].between(2411, 2413)]
events_t = events_df[~events_df['event_id'].between(2411, 2413)]
orders_y = orders_df[orders_df['order_id'].between(101, 120)]
orders_t = orders_df[~orders_df['order_id'].between(101, 120)]
print('events вчера/сегодня:', len(events_y), len(events_t))
print('orders вчера/сегодня:', len(orders_y), len(orders_t))

events вчера/сегодня: 3 384
orders вчера/сегодня: 20 121


## 3.1. Raw-слой: запись CSV, JSON Lines, Parquet (без компрессии) и загрузка в MinIO

Один и тот же DataFrame в трёх представлениях. Метаданные объекта — `source`, `layer`, `format`, `rows`.

In [7]:
def encode_frames(df: pd.DataFrame, parquet_compression: str = None) -> dict:
    """Кодирует один и тот же DataFrame в три представления (bytes).

    :param df: исходный DataFrame.
    :param parquet_compression: сжатие Parquet: None — без сжатия,
        'snappy' / 'zstd' / 'gzip' — со сжатием.
    :return: dict {'csv': bytes, 'json': bytes (JSON Lines), 'parquet': bytes}
    """
    csv_buf = io.BytesIO()
    df.to_csv(csv_buf, index=False)
    json_buf = io.BytesIO()
    df.to_json(json_buf, orient='records', lines=True)  # JSON Lines: одна запись на строку
    pq_buf = io.BytesIO()
    df.to_parquet(pq_buf, engine='pyarrow', index=False, compression=parquet_compression)
    return {'csv': csv_buf.getvalue(), 'json': json_buf.getvalue(), 'parquet': pq_buf.getvalue()}

def upload(source: str, load_date: str, df: pd.DataFrame, parquet_compression: str = None) -> None:
    """Загружает три кодирования df в MinIO в raw-слой.

    Конвенция ключей: raw/{source}/load_date=<DATE>/part-0000.<fmt>.
    В метаданные объекта записываются: source, layer='raw', format, rows.
    """
    enc = encode_frames(df, parquet_compression)
    for fmt, body in enc.items():
        key = f'raw/{source}/load_date={load_date}/part-0000.{fmt}'
        s3.put_object(
            Bucket=BUCKET, Key=key, Body=body,
            ContentType='application/octet-stream',
            Metadata={'source': source, 'layer': 'raw', 'format': fmt, 'rows': str(len(df))},
        )
        print(f's3://{BUCKET}/{key} — {len(body)} байт')

In [8]:
upload('events', YESTERDAY, events_y)
upload('events', TODAY, events_t)
upload('orders', YESTERDAY, orders_y)
upload('orders', TODAY, orders_t)

res = s3.list_objects_v2(Bucket=BUCKET, Prefix='raw/')
print('Объектов в raw/:', res.get('KeyCount'))

s3://de-course/raw/events/load_date=2026-11-04/part-0000.csv — 231 байт


/tmp/ipykernel_155159/1341868395.py:12: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  df.to_json(json_buf, orient='records', lines=True)  # JSON Lines: одна запись на строку
/tmp/ipykernel_155159/1341868395.py:12: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  df.to_json(json_buf, orient='records', lines=True)  # JSON Lines: одна запись на строку


s3://de-course/raw/events/load_date=2026-11-04/part-0000.json — 397 байт
s3://de-course/raw/events/load_date=2026-11-04/part-0000.parquet — 4146 байт
s3://de-course/raw/events/load_date=2026-11-05/part-0000.csv — 20922 байт
s3://de-course/raw/events/load_date=2026-11-05/part-0000.json — 50437 байт
s3://de-course/raw/events/load_date=2026-11-05/part-0000.parquet — 14192 байт
s3://de-course/raw/orders/load_date=2026-11-04/part-0000.csv — 820 байт
s3://de-course/raw/orders/load_date=2026-11-04/part-0000.json — 2082 байт
s3://de-course/raw/orders/load_date=2026-11-04/part-0000.parquet — 4063 байт
s3://de-course/raw/orders/load_date=2026-11-05/part-0000.csv — 4741 байт
s3://de-course/raw/orders/load_date=2026-11-05/part-0000.json — 12669 байт
s3://de-course/raw/orders/load_date=2026-11-05/part-0000.parquet — 6461 байт
Объектов в raw/: 12


/tmp/ipykernel_155159/1341868395.py:12: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  df.to_json(json_buf, orient='records', lines=True)  # JSON Lines: одна запись на строку
/tmp/ipykernel_155159/1341868395.py:12: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  df.to_json(json_buf, orient='records', lines=True)  # JSON Lines: одна запись на строку


### **Контрольная точка 1.** Откройте MinIO Console (`http://localhost:9001`) и найдите бакет. Почему `raw/orders/load_date=.../part-*.parquet` — это не путь к файлу?

## 3.2. Raw-слой: сравнение CSV, JSON, Parquet (без компрессии) размер и чтение одной колонки

Сравниваем **одинаковые данные** (одна партия — «сегодняшние» заказы). Для CSV и JSON «прочитать колонку» = прочитать весь файл (row-based); для Parquet — только нужную колонку из нужного row group. Ориентир по лекции: на больших объёмах Parquet (zstd) в разы компактнее CSV и JSON.

In [9]:
import pyarrow.parquet as pq

def read_csv_bytes(enc_orders: dict, col_name: str):
    """Читает одну колонку из CSV: распарсиваем ВЕСЬ файл (row-based)."""
    return pd.read_csv(io.BytesIO(enc_orders['csv']))[col_name]

def read_json_bytes(enc_orders: dict, col_name: str):
    """Читает одну колонку из JSON Lines: распарсиваем ВЕСЬ файл (row-based)."""
    return pd.read_json(io.BytesIO(enc_orders['json']), lines=True)[col_name]

def read_parquet_one_col(enc_orders: dict, col_name: str):
    """Читает одну колонку из Parquet: только колонка из row group 0 (columnar)."""
    pf = pq.ParquetFile(io.BytesIO(enc_orders['parquet']))
    return pf.read_row_group(0, columns=[col_name]).column(col_name).to_pandas()

def bench(fn, enc_orders: dict, col_name: str, n=5):
    """Запускает fn n раз на одних и тех же данных.

    :return: (минимальное время в мс, результат последнего запуска)
    """
    ts, out = [], None
    for _ in range(n):
        t0 = time.perf_counter()
        out = fn(enc_orders, col_name)
        ts.append((time.perf_counter() - t0) * 1000)
    return min(ts), out

def print_statistic(enc_orders: dict, col_name: str, df: pd.DataFrame) ->  pd.DataFrame:
    results = {}
    for name, fn in [('csv', read_csv_bytes), ('json', read_json_bytes), ('parquet', read_parquet_one_col)]:
        ms, out = bench(fn, enc_orders, col_name)
        assert out is not None and len(out) == len(df), f'{name}: нечитаемый результат'
        results[name] = (len(enc_orders[name]), ms)

    df_res = pd.DataFrame({
        'размер_байт': [results[k][0] for k in results],
        'чтение_1_колонки_мс': [round(results[k][1], 3) for k in results],
    }, index=list(results))

    return df_res


In [10]:
enc_orders = encode_frames(orders_t)  # одна и та же партия во всех трёх форматах
col_name = 'total_amount'  # колонка для сравнения
print_statistic(enc_orders, col_name, orders_t)

/tmp/ipykernel_155159/1341868395.py:12: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  df.to_json(json_buf, orient='records', lines=True)  # JSON Lines: одна запись на строку


,размер_байт,чтение_1_колонки_мс
csv,4741,0.523
json,12669,2.527
parquet,6461,0.169


### **Контрольная точка 2.** На основе вывода статистики определите какой формат компактнее, почему Parquet быстрее именно для **одной** колонки (подсказка: schema-on-write, columnar, skipping)?

## 3.3. Raw-слой: метаданные Parquet (без компрессии)
Метаданные, row groups, min/max — то, на чём работают skipping и выбор одной колонки.

In [11]:
def print_parquet_metadata(enc_orders: dict) -> None:
    """Выводт на экран метаданные файла Parquet."""
    pf = pq.ParquetFile(io.BytesIO(enc_orders['parquet']))
    md = pf.metadata
    print('rows:', md.num_rows, '| row groups:', md.num_row_groups, '| columns:', md.num_columns)
    print('схема:', [f'{f.name}: {f.type}' for f in pf.schema_arrow])
    rg = md.row_group(0)
    for i in range(rg.num_columns):
        c = rg.column(i)
        st = c.statistics
        line = f'{c.path_in_schema:20s} {c.compression:10s}'
        if st and st.has_min_max:
            line += f'  min={st.min}  max={st.max}  nulls={st.null_count}'
        print(line)
    df_rg = pf.read_row_group(0, columns=['order_id'])
    print('прочитано из row group 0:', df_rg.num_rows, 'строк, 1 колонка')

In [12]:
enc_orders = encode_frames(orders_t)
print_parquet_metadata(enc_orders)

rows: 121 | row groups: 1 | columns: 5
схема: ['order_id: int64', 'customer_id: int64', 'order_created_at: timestamp[us]', 'status: large_string', 'total_amount: double']
order_id             UNCOMPRESSED  min=121  max=241  nulls=0
customer_id          UNCOMPRESSED  min=1  max=31  nulls=0
order_created_at     UNCOMPRESSED  min=2026-03-13 10:00:00  max=2026-07-11 10:00:00  nulls=0
status               UNCOMPRESSED  min=cancelled  max=test  nulls=0
total_amount         UNCOMPRESSED  min=450.0  max=2200.0  nulls=0
прочитано из row group 0: 121 строк, 1 колонка


/tmp/ipykernel_155159/1341868395.py:12: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  df.to_json(json_buf, orient='records', lines=True)  # JSON Lines: одна запись на строку


### **Контрольная точка 3.** В Console MinIO: префикс `raw/orders/` → объект → вкладки Key / Metadata. Что хранится в метаданных? Что будет при повторном `put_object` с тем же ключом при **включённом** versioning?

## 4.1. Raw-слой: запись CSV, JSON Lines, Parquet (с компрессией) и загрузка в MinIO

In [13]:

# Решение: 2 партии (events/orders × сегодня), Parquet — со сжатием zstd
upload('events', TODAY, events_t, parquet_compression='zstd')
upload('orders', TODAY, orders_t, parquet_compression='zstd')

res = s3.list_objects_v2(Bucket=BUCKET, Prefix='raw/')
print('Объектов в raw/:', res.get('KeyCount'))

/tmp/ipykernel_155159/1341868395.py:12: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  df.to_json(json_buf, orient='records', lines=True)  # JSON Lines: одна запись на строку


s3://de-course/raw/events/load_date=2026-11-05/part-0000.csv — 20922 байт
s3://de-course/raw/events/load_date=2026-11-05/part-0000.json — 50437 байт
s3://de-course/raw/events/load_date=2026-11-05/part-0000.parquet — 8065 байт
s3://de-course/raw/orders/load_date=2026-11-05/part-0000.csv — 4741 байт
s3://de-course/raw/orders/load_date=2026-11-05/part-0000.json — 12669 байт
s3://de-course/raw/orders/load_date=2026-11-05/part-0000.parquet — 5020 байт
Объектов в raw/: 12


/tmp/ipykernel_155159/1341868395.py:12: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  df.to_json(json_buf, orient='records', lines=True)  # JSON Lines: одна запись на строку


## 4.2. Raw-слой: сравнение CSV, JSON Lines, Parquet (с компрессией) размер и чтение одной колонки

In [14]:
enc_orders = encode_frames(orders_t, parquet_compression='zstd')  # одна и та же партия во всех трёх форматах
col_name = 'total_amount'                                         # колонка для сравнения
print_statistic(enc_orders, col_name, orders_t)

/tmp/ipykernel_155159/1341868395.py:12: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  df.to_json(json_buf, orient='records', lines=True)  # JSON Lines: одна запись на строку


,размер_байт,чтение_1_колонки_мс
csv,4741,0.487
json,12669,2.951
parquet,5020,0.198


## 4.3. Raw-слой: метаданные Parquet (с компрессией)

In [15]:
enc_orders = encode_frames(orders_t, parquet_compression='zstd')
print_parquet_metadata(enc_orders)

rows: 121 | row groups: 1 | columns: 5
схема: ['order_id: int64', 'customer_id: int64', 'order_created_at: timestamp[us]', 'status: large_string', 'total_amount: double']
order_id             ZSTD        min=121  max=241  nulls=0
customer_id          ZSTD        min=1  max=31  nulls=0
order_created_at     ZSTD        min=2026-03-13 10:00:00  max=2026-07-11 10:00:00  nulls=0
status               ZSTD        min=cancelled  max=test  nulls=0
total_amount         ZSTD        min=450.0  max=2200.0  nulls=0
прочитано из row group 0: 121 строк, 1 колонка


/tmp/ipykernel_155159/1341868395.py:12: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  df.to_json(json_buf, orient='records', lines=True)  # JSON Lines: одна запись на строку


### **Контрольная точка 4.** Сравните размер двух файлов Parquet, без компрессии и с компрессией zstd. Что видите?

## 5. Партиционирование

Добавление поля load_date к датафрейму, сохранение таблицы без партиционирования и с партиционированием. Далее — «план чтения» и сравнение планов в DuckDB.

In [16]:
RAW_WO_PART = 'raw_wo_part'     # без партиционирования: load_date — внутри файла
RAW_WITH_PART = 'raw_with_part' # с партиционированием: load_date — в ключе

In [17]:
import numpy as np

orders_p = pd.concat(
    [orders_y.assign(load_date=YESTERDAY),
     orders_t.assign(load_date=TODAY)],
    ignore_index=True,
)
orders_p['load_date'] = np.where(orders_p['order_id'].between(101, 120), YESTERDAY, TODAY)
print(orders_p['load_date'].value_counts())
print('Всего строк:', len(orders_p))
orders_p.head()

load_date
2026-11-05    121
2026-11-04     20
Name: count, dtype: int64
Всего строк: 141


,order_id,customer_id,order_created_at,status,total_amount,load_date
0,101,1,2026-03-01 10:00:00,paid,1000.0,2026-11-04
1,102,1,2026-03-01 10:00:00,paid,900.0,2026-11-04
2,103,1,2026-03-03 11:00:00,paid,900.0,2026-11-04
3,104,2,2026-03-01 11:00:00,paid,1000.0,2026-11-04
4,105,2,2026-03-02 12:00:00,paid,900.0,2026-11-04


In [18]:
# Сохранение без партиционирования
part_keys = []
for shard, part in orders_p.groupby(orders_p['order_id'] % 3):
    buf = io.BytesIO()
    part.to_parquet(buf, engine='pyarrow', index=False)
    key = f'{RAW_WO_PART}/orders/part-{shard:04d}.parquet'
    s3.put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue())
    part_keys.append(key)
    print(f's3://{BUCKET}/{key} — {len(buf.getvalue())} байт')
print(f'Загружено в {RAW_WO_PART}/: {part_keys}')

s3://de-course/raw_wo_part/orders/part-0000.parquet — 5102 байт
s3://de-course/raw_wo_part/orders/part-0001.parquet — 5064 байт
s3://de-course/raw_wo_part/orders/part-0002.parquet — 5053 байт
Загружено в raw_wo_part/: ['raw_wo_part/orders/part-0000.parquet', 'raw_wo_part/orders/part-0001.parquet', 'raw_wo_part/orders/part-0002.parquet']


In [19]:
# Сохранение С партиционированием: load_date — в имени папки, из файла колонка убирается
part_keys = []
for load_date, part in orders_p.groupby('load_date', sort=False):
    part = part.drop(columns=['load_date'])
    buf = io.BytesIO()
    part.to_parquet(buf, engine='pyarrow', index=False)
    key = f'{RAW_WITH_PART}/orders/load_date={load_date}/part-0000.parquet'
    s3.put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue())
    part_keys.append(key)
    print(f's3://{BUCKET}/{key} — {len(buf.getvalue())} байт')
print(f'Загружено в {RAW_WITH_PART}/: {part_keys}')

s3://de-course/raw_with_part/orders/load_date=2026-11-04/part-0000.parquet — 3958 байт
s3://de-course/raw_with_part/orders/load_date=2026-11-05/part-0000.parquet — 5685 байт
Загружено в raw_with_part/: ['raw_with_part/orders/load_date=2026-11-04/part-0000.parquet', 'raw_with_part/orders/load_date=2026-11-05/part-0000.parquet']


In [20]:
# «План чтения» выборки за вчера: какие файлы придётся открыть
def plan_files(prefix: str) -> list:
    """Перечисляет все ключи объектов в MinIO по префиксу (с пагинацией)."""
    files, token = [], None
    while True:
        r = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix, **({'ContinuationToken': token} if token else {}))
        files += [o['Key'] for o in r.get('Contents', [])]
        if r.get('IsTruncated'):
            token = r['NextContinuationToken']
        else:
            return files

plan_A = plan_files(f'{RAW_WO_PART}/orders/')                           # без партиционирования: датного префикса нет
plan_B = plan_files(f'{RAW_WITH_PART}/orders/load_date={YESTERDAY}/')   # с партиционированием: только префикс за вчера
print(f'Без партиционирования: {len(plan_A)} файл(а) — нужно открыть ВСЕ и фильтровать внутри')
print(f'С партиционированием: {len(plan_B)} файл(а) — partition pruning по префиксу')
for k in plan_B:
    print('  ', k)

Без партиционирования: 3 файл(а) — нужно открыть ВСЕ и фильтровать внутри
С партиционированием: 1 файл(а) — partition pruning по префиксу
   raw_with_part/orders/load_date=2026-11-04/part-0000.parquet


In [21]:
%pip install duckdb

import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs")

con.execute(f"""
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id={S3_USER};
    SET s3_secret_access_key={S3_PASSWORD};
    SET s3_url_style='path';
    SET s3_use_ssl=false;
""")

Note: you may need to restart the kernel to use updated packages.


In [22]:
# Без партиционирования: load_date — обычная колонка данных, фильтр строки по строкам
con.execute(f"""
    SELECT *
    FROM 's3://de-course/raw_wo_part/orders/*.parquet'
    WHERE load_date = date('{YESTERDAY}')
""").df()

,order_id,customer_id,order_created_at,status,total_amount,load_date
0,102,1,2026-03-01 10:00:00,paid,900.0,2026-11-04
1,105,2,2026-03-02 12:00:00,paid,900.0,2026-11-04
2,108,4,2026-03-04 15:00:00,pending,100.0,2026-11-04
3,111,6,2026-03-06 18:00:00,paid,100.0,2026-11-04
4,114,7,2026-03-08 20:00:00,paid,1000.0,2026-11-04
5,117,9,2026-03-10 12:00:00,paid,800.0,2026-11-04
6,120,10,2026-03-12 15:00:00,paid,900.0,2026-11-04
7,103,1,2026-03-03 11:00:00,paid,900.0,2026-11-04
8,106,3,2026-03-03 13:00:00,paid,900.0,2026-11-04
9,109,4,2026-03-05 16:00:00,cancelled,200.0,2026-11-04


In [23]:
# С партиционированием: load_date извлекается из имени пути (hive-партиционирование)
con.execute(f"""
    SELECT *
    FROM 's3://de-course/raw_with_part/orders/**/*.parquet'
    WHERE load_date = date('{YESTERDAY}')
""").df()

,order_id,customer_id,order_created_at,status,total_amount,load_date
0,101,1,2026-03-01 10:00:00,paid,1000.0,2026-11-04
1,102,1,2026-03-01 10:00:00,paid,900.0,2026-11-04
2,103,1,2026-03-03 11:00:00,paid,900.0,2026-11-04
3,104,2,2026-03-01 11:00:00,paid,1000.0,2026-11-04
4,105,2,2026-03-02 12:00:00,paid,900.0,2026-11-04
5,106,3,2026-03-03 13:00:00,paid,900.0,2026-11-04
6,107,3,2026-03-04 14:00:00,returned,700.0,2026-11-04
7,108,4,2026-03-04 15:00:00,pending,100.0,2026-11-04
8,109,4,2026-03-05 16:00:00,cancelled,200.0,2026-11-04
9,110,5,2026-03-05 17:00:00,paid,700.0,2026-11-04


In [24]:
# План: без партиционирования — ожидаем PARQUET_SCAN + отдельный FILTER по строкам
plan = con.execute(f"""
    EXPLAIN
    SELECT *
    FROM 's3://de-course/raw_wo_part/orders/*.parquet'
    WHERE load_date = date('{YESTERDAY}')
""").df()
print(plan.to_markdown(index=False))

ImportError: `Import tabulate` failed.  Use pip, conda, or your preferred package management tool to install the tabulate package.

In [ ]:
sudo apt install -y tabulate

SyntaxError: invalid syntax (3038279860.py, line 1)

In [ ]:
# План: с партиционированием — ожидаем File Filters и Scanning Files 1/2
plan = con.execute(f"""
    EXPLAIN
    SELECT *
    FROM 's3://de-course/raw_with_part/orders/**/*.parquet'
    WHERE load_date = date('{YESTERDAY}')
""").df()
print(plan.to_markdown(index=False))

ImportError: `Import tabulate` failed.  Use pip, conda, or your preferred package management tool to install the tabulate package.

### **Контрольная точка 5.** Посмотрите планы. В чём разница? Какой вариант эффективнее и почему?

## 6. Delta Lake (если есть время)

DuckDB + Delta-расширение: пишем таблицу Delta (Parquet-данные + журнал `_delta_log`) в MinIO и читаем обратно

In [25]:
%pip install -q deltalake

Note: you may need to restart the kernel to use updated packages.


In [26]:
import os
from deltalake import write_deltalake, DeltaTable

# Переопределяем окружение, чтобы клиент не полез в чужой эндпоинт (иначе возможна ошибка обращения к s3.local.amazonaws.com)
os.environ["AWS_ENDPOINT_URL"] = S3_ENDPOINT
os.environ["AWS_ENDPOINT_URL_S3"] = S3_ENDPOINT
os.environ["AWS_ACCESS_KEY_ID"] = S3_USER
os.environ["AWS_SECRET_ACCESS_KEY"] = S3_PASSWORD
os.environ["AWS_REGION"] = "us-east-1"

S3_STORAGE_OPTIONS = {
    "aws_endpoint": S3_ENDPOINT,
    "aws_access_key_id": S3_USER,
    "aws_secret_access_key": S3_PASSWORD,
    "aws_region": "us-east-1",
    "aws_allow_http": "true",
    "aws_virtual_hosted_style_request": "false",
}

prefix = "raw_delta/orders"

In [27]:
# df — результат SELECT через DuckDB
df = con.execute(f"""
    SELECT order_id
         , customer_id
         , order_created_at
         , status
         , total_amount
         , date(load_date) as load_date
    FROM 's3://de-course/raw_with_part/orders/**/*.parquet'
    WHERE load_date = date('{YESTERDAY}')
""").df()
df.head()

,order_id,customer_id,order_created_at,status,total_amount,load_date
0,101,1,2026-03-01 10:00:00,paid,1000.0,2026-11-04
1,102,1,2026-03-01 10:00:00,paid,900.0,2026-11-04
2,103,1,2026-03-03 11:00:00,paid,900.0,2026-11-04
3,104,2,2026-03-01 11:00:00,paid,1000.0,2026-11-04
4,105,2,2026-03-02 12:00:00,paid,900.0,2026-11-04


In [28]:

# Сохраняем df как таблицу Delta Lake
delta_table_path = f"s3://{BUCKET}/{prefix}"

print(f"Выполним сохранение таблицы {delta_table_path} 2 раза подряд")
for i in (0, 1):
    write_deltalake(
        table_or_uri=delta_table_path,
        data=df,
        storage_options=S3_STORAGE_OPTIONS,
        partition_by=["load_date"],
        mode="overwrite",
    )
    print(f"{i}: запись завершена")

Выполним сохранение таблицы s3://de-course/raw_delta/orders 2 раза подряд
0: запись завершена
1: запись завершена


In [29]:
dt = DeltaTable(table_uri=f"s3://{BUCKET}/{prefix}", storage_options=S3_STORAGE_OPTIONS)
history = dt.history(limit=10)  # последние 10 коммитов, limit не нужен — все

# Каждая запись — один коммит (CREATE, WRITE, DELETE, MERGE, OPTIMIZE, REWIND...)
for v in history:
    print(v["version"], v["timestamp"], v["operation"],
          v.get("operationParameters", {}).get("partitionFilters", ""))

# print(*dt.history(), sep='\n')

1 1793887028438 WRITE 
0 1793887027788 WRITE 


In [30]:
# Откатить таблицу к состоянию на момент версии 0
dt.restore(0)

{'numRemovedFile': 1, 'numRestoredFile': 1}

In [ ]:
# Дерево в бакете: файлы данных + _delta_log
res = s3.list_objects_v2(Bucket=BUCKET, Prefix=f'{prefix}/')
for o in res.get('Contents', []):
    print(o['Key'], o['Size'], 'байт')

### **Контрольная точка 6.** Что даёт папка `_delta_log`, чего нет у обычного каталога с Parquet-файлами?

## Финал

1. Сохранить ноутбук.
2. В терминале: `docker compose down` (тома не удаляем, чтобы не потерять raw-слой).